### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Step 2: Open the project directory

Replace `Your_Dir` with your own path.

In [5]:
%cd /content/drive/MyDrive/

!git clone https://github.com/KianBaghai/emg2qwerty

/content/drive/MyDrive
Cloning into 'emg2qwerty'...
remote: Enumerating objects: 269, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 269 (delta 12), reused 9 (delta 9), pack-reused 252 (from 3)
Receiving objects: 100% (269/269), 33.64 MiB | 18.59 MiB/s, done.
Resolving deltas: 100% (109/109), done.
Updating files: 100% (85/85), done.
Filtering content: 100% (17/17), 1.00 GiB | 26.42 MiB/s, done.
fatal: cannot exec '/content/drive/MyDrive/emg2qwerty/.git/hooks/post-checkout': Permission denied


In [14]:
%cd /content/drive/MyDrive/emg2qwerty/

!chmod +x .git/hooks/post-merge

!git pull
!git checkout transformers

/content/drive/MyDrive/emg2qwerty
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 1.07 KiB | 33.00 KiB/s, done.
From https://github.com/KianBaghai/emg2qwerty
   88d10eb..53a51f5  transformers -> origin/transformers
Updating 88d10eb..53a51f5
Fast-forward
 config/model/transformer_ctc.yaml | 3 +--
 emg2qwerty/lightning.py           | 1 -
 emg2qwerty/modules.py             | 3 +--
 3 files changed, 2 insertions(+), 5 deletions(-)
M	scripts/lm/build_char_lm.sh
Already on 'transformers'
Your branch is up to date with 'origin/transformers'.
fatal: cannot exec '.git/hooks/post-checkout': Permission denied


### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [3]:
!pip install -r requirements.txt

  Using cached https://github.com/kpu/kenlm/archive/master.zip (553 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


### Step 4: Start your experiments!

- Remember to download and copy the dataset to this directory: `Your_Dir/emg2qwerty/data`.
- You may now start your experiments with any scripts! Below are examples of single-user training and testing (greedy decoding).
- **There are two ways to track the logs:**
  - 1. Keep `--multirun`, and the logs will not be printed here, but they will be saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/submitit_logs/`.
  - 2. Comment out `--multirun` and the logs will be printed in this notebook, but they will not be saved.

#### Training

- The checkpoints are saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/checkpoints/`.

In [15]:
!python -m emg2qwerty.train model=transformer_ctc user=single_user trainer.max_epochs=60

[2026-03-12 21:24:02,121][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

#### Testing:

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.

In [ ]:
# Single-user testing
!python -m emg2qwerty.train \
  user="single_user" \
  checkpoint="Your_Path_to_Checkpoint" \
  train=False trainer.accelerator=gpu \
  decoder=ctc_greedy \
  hydra.launcher.mem_gb=64 \
  # --multirun